In [ ]:
# %matplotlib ipympl
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import LightSource
import matplotlib.animation as anim

In [ ]:
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Times']})
rc('text', usetex=True)

In [ ]:
n_qubits = 10
graph_num = 1
qubitstate_size = 2**n_qubits
filepath = str(n_qubits) + "q_" + str(graph_num) + "/"
savefilename = filepath[:-1] + ".mp4"
fps = 40

_x, _y, _z = np.arange(0,n_qubits,1), np.arange(0,n_qubits,1), np.arange(0,n_qubits,1)
_xx, _yy, __zz = np.meshgrid(_x, _y, _z, indexing="ij")
x, y, z = _xx.ravel(), _yy.ravel(), __zz.ravel()
u, v = np.mgrid[0:2*np.pi:50j, 0:np.pi:50j]

In [ ]:
light = LightSource(0,20)
fig = plt.figure(figsize=(4,3),dpi=300)
ax = fig.add_subplot(projection='3d')
ax.view_init(elev=25,azim=-130)
ax.set_xlabel(r'$m$')
ax.set_ylabel(r'$n$')
ax.set_xlim(0,n_qubits)
ax.set_ylim(0,n_qubits)
ax.set_zlim(0,n_qubits)

In [ ]:
class sphere:
    def __init__(self, x, y, z, r):
        self.x = x
        self.y = y
        self.z = z
        self.r = r

In [6]:
sphere_lists = []
surfaces = []

txt = ax.text(0.9, 0.9, 0.9, "(mu,nu) =" + "(" + str(0) + "," + str(0) + ")", transform=ax.transAxes)

for m in range(qubitstate_size):
    for n in range(qubitstate_size):
        filename = str(m) + "," + str(n) + ".txt"
        symQfunc = np.genfromtxt("C:/dev/StabilizerStates/data/symQfuncs/"+filepath+filename, usecols = 0, delimiter=",", dtype = float)
        symQfunc = np.reshape(symQfunc,(n_qubits+1,n_qubits+1,n_qubits+1))

        maxQ = np.max(symQfunc)
        sphere_list = []
        for (i,j,k) in zip(x,y,z):
            r = symQfunc[i,j,k] / maxQ
            if r > 0.05:
                sphere_list.append(sphere(i, j, k, r))
        sphere_lists.append(sphere_list)
                

KeyboardInterrupt: 

In [ ]:
def update_plot(frame):
    N = len(surfaces)
    for j in range(N):
        surfaces[0].remove()
        surfaces.pop(0)
    for sphere in sphere_lists[frame]:
        s_x = sphere.r*np.cos(u)*np.sin(v)
        s_y = sphere.r*np.sin(u)*np.sin(v)
        s_z = sphere.r*np.cos(v)
        sp = ax.plot_surface(s_x+sphere.x, s_y+sphere.y, s_z+sphere.z, color = 'b', alpha=0.5)
        surfaces.append(sp)
    nu = frame % qubitstate_size
    mu = int( (frame - nu) / qubitstate_size)
    txt.set_text("(mu,nu) =" + "(" + str(mu) + "," + str(nu) + ")")
    return surfaces, txt

In [ ]:
ani = anim.FuncAnimation(fig, update_plot, frames = qubitstate_size**2, interval = 100)
ani.save(savefilename, writer=anim.FFMpegWriter(fps=fps))